# Enhanced Federated Learning Cycle for DeepFake Detection (Kaggle)

This notebook is tailored for Kaggle execution with optional GPU acceleration.

Pipeline modules used:
- enhanced_client_selection.py
- update_validation.py
- knowledge_distillation.py
- client_reputation_ledger.py
- evaluation_metrics.py
- flwr_federated_cycle.py

In [ ]:
# 0) Kaggle bootstrap: clone project repo into writable working directory
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Update if needed.
REPO_URL = "https://github.com/AverageWeebo101/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection.git"
REPO_BRANCH = "CodespaceVariant"
CLONE_DIR = Path("/kaggle/working/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection")

if not CLONE_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CLONE_DIR)])
    print(f"Cloned repository to: {CLONE_DIR}")
else:
    print(f"Repository already exists at: {CLONE_DIR}")
    # Keep the local copy fresh for reruns.
    subprocess.check_call(["git", "-C", str(CLONE_DIR), "fetch", "origin", REPO_BRANCH, "--depth", "1"])
    subprocess.check_call(["git", "-C", str(CLONE_DIR), "checkout", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CLONE_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"])
    print(f"Updated local clone to origin/{REPO_BRANCH}")

# Ensure imports resolve from the cloned repo and relative paths behave as expected.
if str(CLONE_DIR) not in sys.path:
    sys.path.insert(0, str(CLONE_DIR))
os.chdir(CLONE_DIR)

print(f"Working directory: {Path.cwd()}")

In [ ]:
# 1) Install dependencies and import Flower pipeline modules (run Cell 2 first)
import importlib
import importlib.util
import subprocess
import sys

def _pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"pip install failed: {' '.join(packages)}")

def _ensure_package(import_name: str, pip_spec: str) -> None:
    if importlib.util.find_spec(import_name) is None:
        _pip_install(pip_spec)

# Keep TensorFlow GPU-capable on Kaggle (avoid tensorflow-cpu).
_ensure_package("tensorflow", "tensorflow>=2.15")
_ensure_package("ray", "ray[default]>=2.9")
_ensure_package("flwr", "flwr[simulation]>=1.7")

if "flwr_federated_cycle" in sys.modules:
    importlib.reload(sys.modules["flwr_federated_cycle"])

from flwr_federated_cycle import (
    FLWRCycleConfig,
    FLWRFederatedLearningCycle,
)

print("Flower environment is ready.")

## Configuration

Stability-first baseline for Kaggle Ray. Scale up only after successful rounds.

In [ ]:
# 2) Kaggle GPU setup
import tensorflow as tf

USE_KAGGLE_GPU = True
gpus = tf.config.list_physical_devices("GPU")

if USE_KAGGLE_GPU and gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {len(gpus)} device(s).")
else:
    print("No GPU detected (or GPU disabled). Training will run on CPU.")

In [ ]:
# 3) Kaggle-focused FL configuration (stability-first baseline)
from knowledge_distillation import DistillationConfig
from enhanced_client_selection import SelectionWeights
from update_validation import ContributionWeights, ClippingConfig
from client_reputation_ledger import ReputationConfig

HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0

# Kept for parity notes with the old TFF pipeline.
TFF_ONLY_SETTINGS = {
    "enable_comparison": False,
}

# Phase 1 profile: prioritize stable rounds on Kaggle Ray.
# Scale up later only after this profile completes multiple rounds without worker crashes.
config = FLWRCycleConfig(
    # Core FL settings
    model_path="efficientnetb4_final.keras",
    num_devices=20,
    local_epochs=1,
    global_rounds=3,
    clients_per_round=5,
    local_batch_size=8,
    local_lr=1e-4,
    eval_every=1,

    # Knowledge Distillation (Part 3)
    # Keep disabled during stabilization; re-enable after baseline stability is confirmed.
    enable_distillation=False,
    distillation_config=DistillationConfig(
        temperature=2.0,
        lam=0.5,
        epochs=3,
        batch_size=32,
        learning_rate=1e-4,
    ),

    # Client Selection Weights (Part 1)
    selection_weights=SelectionWeights(
        w_v=0.30,
        w_d=0.20,
        w_l=0.10,
        w_r=0.25,
        w_s=0.15,
    ),

    # Contribution Weights (Part 2)
    contribution_weights=ContributionWeights(
        alpha=0.35,
        beta=0.20,
        gamma=0.20,
        delta=0.25,
    ),
    clipping_config=ClippingConfig(
        clip_threshold=10.0,
        clip_value=5.0,
    ),
    harmful_threshold=0.02,

    # Reputation Ledger (Part 4)
    reputation_config=ReputationConfig(
        theta=0.0,
        gamma=0.10,
        decay_rate=0.99,
        floor=0.05,
        ceiling=1.0,
        initial_reputation=0.50,
        penalty_factor=0.05,
    ),

    # Flower simulation/output settings
    reports_dir="reports",
    checkpoints_dir="reports/checkpoints_flwr",
    auto_resume_from_checkpoint=True,
    simulation_client_cpus=1.0,
    simulation_client_gpus=0.0,
    simulation_local_mode=False,
    tflite_output_path="effnet_global_flwr_final.tflite",
)

RAMP_GUIDE = [
    "After stable 3 rounds: clients_per_round 5 -> 7 -> 10",
    "Then try local_batch_size 8 -> 12 -> 16 (only if still stable)",
    "Then increase global_rounds/local_epochs",
    "Re-enable distillation last",
]

print("Configuration created (stability profile).")
print(f"  GPU detected:     {HAS_GPU}")
print(f"  Devices:          {config.num_devices}")
print(f"  Rounds:           {config.global_rounds}")
print(f"  Local epochs:     {config.local_epochs}")
print(f"  Clients/round:    {config.clients_per_round}")
print(f"  Batch size:       {config.local_batch_size}")
print(f"  Distillation:     {config.enable_distillation}")
print(f"  Ray cpus/client:  {config.simulation_client_cpus}")
print(f"  Ray gpus/client:  {config.simulation_client_gpus}")
print(f"  Ray local_mode:   {config.simulation_local_mode}")
print(f"  TFF comparison:   {TFF_ONLY_SETTINGS['enable_comparison']} (not applicable in Flower)")
print("Ramp plan:")
for step in RAMP_GUIDE:
    print(f"  - {step}")

In [ ]:
# Optional: download TFRecords from Google Drive into Kaggle working dir
import os
import glob
import subprocess
import sys

GOOGLE_DRIVE_TFRECORD_FOLDER_URL = "https://drive.google.com/drive/folders/1BFevyQRFdtinNTjhUtLnoVsWDsDiwUEE?usp=sharing"  # paste folder URL here
DOWNLOAD_ROOT = "/kaggle/working/ffpp_tfrecord_clients"
TFRECORD_GLOB = "client_*.tfrecord"

def ensure_drive_tfrecords() -> str:
    if os.path.isdir(DOWNLOAD_ROOT) and glob.glob(os.path.join(DOWNLOAD_ROOT, TFRECORD_GLOB)):
        print(f"Using existing TFRecords in: {DOWNLOAD_ROOT}")
        return DOWNLOAD_ROOT

    if not GOOGLE_DRIVE_TFRECORD_FOLDER_URL.strip():
        raise RuntimeError("Set GOOGLE_DRIVE_TFRECORD_FOLDER_URL or provide local Kaggle dataset path.")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown>=5.0"])
    os.makedirs(DOWNLOAD_ROOT, exist_ok=True)

    # Download entire shared Drive folder
    subprocess.check_call([
        "gdown",
        "--folder",
        GOOGLE_DRIVE_TFRECORD_FOLDER_URL.strip(),
        "-O",
        DOWNLOAD_ROOT,
    ])

    files = glob.glob(os.path.join(DOWNLOAD_ROOT, TFRECORD_GLOB))
    if not files:
        raise RuntimeError("Download finished but no TFRecord files were found.")
    print(f"Downloaded {len(files)} TFRecord shards to: {DOWNLOAD_ROOT}")
    return DOWNLOAD_ROOT

KAGGLE_TFRECORD_ROOT = ensure_drive_tfrecords()

In [ ]:
# 4) Load FF++ TFRecords from Kaggle working dir or dataset mount
import glob
import os
from pathlib import Path

import tensorflow as tf

TFRECORD_GLOB = "client_*.tfrecord"
COMPRESSION_TYPE = "GZIP"
VAL_CLIENTS = 10
TEST_CLIENTS = 10
SHUFFLE_BUFFER = 2048

# Preferred search order:
# 1) Existing KAGGLE_TFRECORD_ROOT from previous cells (e.g., Google Drive download cell)
# 2) Kaggle writable working directory
# 3) Kaggle input dataset mount
DEFAULT_WORKING_ROOT = "/kaggle/working/ffpp_tfrecord_clients"
DEFAULT_INPUT_ROOT = "/kaggle/input/ff-c23-tfrecord/ffpp_tfrecord_clients"


def _resolve_model_input_shape() -> tuple[int, int]:
    try:
        from tensorflow.keras.applications.efficientnet import preprocess_input as _effnet_preprocess

        model = tf.keras.models.load_model(
            config.model_path,
            compile=False,
            custom_objects={"preprocess_input": _effnet_preprocess},
        )
        input_shape = model.input_shape
        if isinstance(input_shape, list):
            input_shape = input_shape[0]

        height = int(input_shape[1])
        width = int(input_shape[2])
        channels = int(input_shape[3]) if input_shape[3] is not None else 3

        config.input_shape = (height, width, channels)
        return (height, width)
    except Exception as exc:
        print(f"Warning: could not infer model input shape ({exc}). Using config.input_shape={config.input_shape}.")
        return tuple(config.input_shape[:2])


def _has_tfrecord_shards(root: str) -> bool:
    return bool(root and os.path.isdir(root) and glob.glob(os.path.join(root, TFRECORD_GLOB)))


def _resolve_tfrecord_root() -> str:
    candidates = []

    existing_root = globals().get("KAGGLE_TFRECORD_ROOT", "")
    if isinstance(existing_root, str) and existing_root.strip():
        candidates.append(existing_root.strip())

    candidates.append(DEFAULT_WORKING_ROOT)
    candidates.append(DEFAULT_INPUT_ROOT)

    # Auto-discover any TFRecord directory under /kaggle/working if needed.
    for p in Path("/kaggle/working").glob("**/client_*.tfrecord"):
        candidates.append(str(p.parent))

    seen = set()
    for root in candidates:
        if root in seen:
            continue
        seen.add(root)
        if _has_tfrecord_shards(root):
            return root

    raise FileNotFoundError(
        "Could not find TFRecords in Kaggle working directory or kaggle/input mount. "
        "Run the Google Drive download cell first or set KAGGLE_TFRECORD_ROOT manually."
    )


def parse_example(example_proto: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    feature_desc = {
        "image/encoded": tf.io.FixedLenFeature([], tf.string),
        "image/format": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.float32),
    }
    parsed = tf.io.parse_single_example(example_proto, feature_desc)

    image = tf.io.decode_jpeg(parsed["image/encoded"], channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    label = tf.cast(parsed["label"], tf.float32)
    return image, label


def load_client_dataset(tfrecord_path: str) -> tf.data.Dataset:
    ds = tf.data.TFRecordDataset(
        tfrecord_path,
        compression_type=COMPRESSION_TYPE,
        num_parallel_reads=tf.data.AUTOTUNE,
    )
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    return ds


def concat_datasets(dataset_list: list[tf.data.Dataset]) -> tf.data.Dataset:
    if not dataset_list:
        raise ValueError("No datasets provided for concatenation.")
    combined = dataset_list[0]
    for ds in dataset_list[1:]:
        combined = combined.concatenate(ds)
    return combined


IMG_SIZE = _resolve_model_input_shape()
KAGGLE_TFRECORD_ROOT = _resolve_tfrecord_root()

all_files = sorted(glob.glob(os.path.join(KAGGLE_TFRECORD_ROOT, TFRECORD_GLOB)))
if len(all_files) < 3:
    raise RuntimeError(
        f"Expected multiple TFRecord client shards at {KAGGLE_TFRECORD_ROOT}, found {len(all_files)}"
    )

n_total = len(all_files)
n_val = min(VAL_CLIENTS, max(1, n_total // 10))
n_test = min(TEST_CLIENTS, max(1, n_total // 10))

val_files = all_files[:n_val]
test_files = all_files[n_val:n_val + n_test]
train_files = all_files[n_val + n_test:]

if not train_files:
    raise RuntimeError("No train TFRecords left after val/test split. Reduce VAL_CLIENTS/TEST_CLIENTS.")

selected_train_files = train_files[: config.num_devices]

# IMPORTANT: pass serializable client specs (paths) to Flower/Ray.
client_data = {
    str(i): fp
    for i, fp in enumerate(selected_train_files)
}

config.num_devices = len(client_data)
server_val_data = concat_datasets([load_client_dataset(fp) for fp in val_files])
test_data = concat_datasets([load_client_dataset(fp) for fp in test_files])
proxy_data = concat_datasets([
    load_client_dataset(fp).map(lambda image, label: image, num_parallel_calls=tf.data.AUTOTUNE)
    for fp in selected_train_files
])

print(f"TFRecord root: {KAGGLE_TFRECORD_ROOT}")
print(f"Image size used: {IMG_SIZE}")
print(f"Total shards: {len(all_files)}")
print(f"Train/Val/Test shards: {len(selected_train_files)}/{len(val_files)}/{len(test_files)}")
print(f"Using num_devices={config.num_devices}")

In [ ]:
# 5) Build and run Flower cycle
import importlib
import sys

# Ensure we run the latest cloned version of the module from /kaggle/working.
if "flwr_federated_cycle" in sys.modules:
    importlib.reload(sys.modules["flwr_federated_cycle"])

print("Run profile:")
print(f"  devices={config.num_devices}, rounds={config.global_rounds}, local_epochs={config.local_epochs}")
print(f"  clients_per_round={config.clients_per_round}, batch={config.local_batch_size}")
print(f"  cpus/client={config.simulation_client_cpus}, gpus/client={config.simulation_client_gpus}, local_mode={config.simulation_local_mode}")
print(f"  distillation={config.enable_distillation}")

cycle = FLWRFederatedLearningCycle(config)
cycle.load_global_model()
cycle.create_clients(client_data)
cycle.setup_enhancement_modules()

history = cycle.run(
    server_val_data=server_val_data,
    test_data=test_data,
    proxy_data=proxy_data,
)

print("Training complete.")
print(f"Rounds completed: {len(history.get('round', []))}")

In [ ]:
# 6) Quick summary
if history.get("enhanced_accuracy"):
    best_acc = max(history["enhanced_accuracy"])
    final_acc = history["enhanced_accuracy"][-1]
    print(f"Best enhanced accuracy:  {best_acc:.4f}")
    print(f"Final enhanced accuracy: {final_acc:.4f}")

print(f"TFLite output: {config.tflite_output_path}")
print("Reports dir: reports/")